In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the data we already fetched
actuals = pd.read_csv('data/actuals_jan.csv')
forecasts = pd.read_csv('data/forecasts_jan.csv')

# Convert to datetime
actuals['startTime'] = pd.to_datetime(actuals['startTime'])
forecasts['startTime'] = pd.to_datetime(forecasts['startTime'])
forecasts['publishTime'] = pd.to_datetime(forecasts['publishTime'])

# 2. Merge data on startTime
# We keep only the latest forecast for each startTime
latest_forecasts = forecasts.sort_values('publishTime').groupby('startTime').last().reset_index()
df = pd.merge(actuals, latest_forecasts, on='startTime', suffixes=('_act', '_for'))

# 3. Calculate Error Metrics
df['abs_error'] = abs(df['generation_act'] - df['generation_for'])
df['hour'] = df['startTime'].dt.hour

print(f"Mean Absolute Error (MAE): {df['abs_error'].mean():.2f} MW")
print(f"Median Absolute Error: {df['abs_error'].median():.2f} MW")
print(f"P99 Error (99th Percentile): {df['abs_error'].quantile(0.99):.2f} MW")

# 4. Subjective Reliability Recommendation
# Question: How many MW can we reliably expect?
# Logic: We look at the 5th percentile of actual generation (the amount available 95% of the time)
reliable_mw = df['generation_act'].quantile(0.05)

print(f"\n--- RELIABILITY RECOMMENDATION ---")
print(f"Based on Jan 2024 data, we can reliably expect {reliable_mw:.2f} MW.")
print(f"Reasoning: This value represents the 5th percentile of generation, meaning actual")
print(f"wind output remained above this level 95% of the time during the analysis period.")